# 문장 -> 벡터(1차원 숫자 배열 [8.1,9.1, 2, 5, 4, 3....])

- openAi API : https://platform.openai.com/ 의 키를 .env등록
- upstage : https://console.upstage.ai/ 의 키를 등록

# 1. 환경변수 load

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

# 2. 유사도 계산하는 방법(https://www.pinecone.io/learn/vector-similarity)

     1. 유클리드 거리 : 두 벡터 사이의 직선 거리(거리가 얼마나 가까운지)
     2. 코사인 유사도 : 두 벡터 사이의 각도(방향이 얼마나 유사한지)
     dot product : 두 벡터의 내적(크기와 방향이 얼마나 유사한지)


In [5]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1) 
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)


# 3. openAI API의 embedding model 사용

In [8]:
from openai import OpenAI
openai_client = OpenAI()

In [12]:
# text-embedding-3-large

response = openai_client.embeddings.create(
    input="king",
    model="text-embedding-3-large"
)

In [18]:
import numpy as np
king_vector = np.array(response.data[0].embedding)
print(king_vector.shape)
print(king_vector)


(3072,)
[ 0.01040417  0.02499519 -0.0014776  ...  0.00835009  0.01049861
 -0.00254005]


In [19]:
queen_response = openai_client.embeddings.create(
    input="queen",
    model="text-embedding-3-large"
)

In [22]:
queen_vector = np.array(queen_response.data[0].embedding)
print(queen_vector.shape)
print(queen_vector)

(3072,)
[-0.0138568   0.00087419 -0.01678164 ...  0.00018065  0.01159801
  0.00642524]


In [24]:
# 유사도 검사
king_queen_similarity = cosine_similarity(king_vector, queen_vector)
print(f"King과 Queen의 유사도: {king_queen_similarity:.4f}")


King과 Queen의 유사도: 0.5552


In [25]:
slave_response = openai_client.embeddings.create(
    input="slave",
    model="text-embedding-3-large"
)
slave_vector = np.array(slave_response.data[0].embedding)
print(slave_vector.shape)
print(slave_vector)

(3072,)
[-0.01999537  0.00620363  0.01191717 ...  0.00094749 -0.02679118
 -0.0058524 ]


In [26]:
king_slave_similarity = cosine_similarity(king_vector, slave_vector)
print(f"King과 Slave의 유사도: {king_slave_similarity:.4f}")

King과 Slave의 유사도: 0.2948


In [ ]:
# 한국어 예시

In [28]:
kor_king_response = openai_client.embeddings.create(
    input="왕",
    model="text-embedding-3-large"
)
kor_king_vector = np.array(kor_king_response.data[0].embedding)
print(kor_king_vector.shape)
print(kor_king_vector)

(3072,)
[-0.00595223  0.01159333 -0.01316932 ... -0.00357134  0.01323696
 -0.00083999]


In [29]:
kor_queen_response = openai_client.embeddings.create(
    input="여왕",
    model="text-embedding-3-large"
)
kor_queen_vector = np.array(kor_queen_response.data[0].embedding)
print(kor_queen_vector.shape)
print(kor_queen_vector)

(3072,)
[-0.01307151 -0.00921458 -0.00532257 ... -0.00482468 -0.00204418
  0.02035061]


In [30]:
# 왕과 여왕의 유사도
kor_king_queen_similarity = cosine_similarity(kor_king_vector, kor_queen_vector)
print(f"왕과 여왕의 유사도: {kor_king_queen_similarity:.4f}")

왕과 여왕의 유사도: 0.4873


In [31]:
kor_slave_response = openai_client.embeddings.create(
    input="거지",
    model="text-embedding-3-large"
)
kor_slave_vector = np.array(kor_slave_response.data[0].embedding)
print(kor_slave_vector.shape)
print(kor_slave_vector)

(3072,)
[-0.02400834 -0.02815736 -0.00371585 ...  0.01028707 -0.00947125
  0.03754314]


In [32]:
# 왕과 거지의 유사도
kor_king_slave_similarity = cosine_similarity(kor_king_vector, kor_slave_vector)
print(f"왕과 거지의 유사도: {kor_king_slave_similarity:.4f}")

왕과 거지의 유사도: 0.2552


In [34]:
# king과 왕의 유사도
king_kor_similarity = cosine_similarity(king_vector, kor_king_vector)
print(f"King과 왕의 유사도: {king_kor_similarity:.4f}")

King과 왕의 유사도: 0.5475


# 4. upstage의 embedding model 사용
- 한국에 embedding에는 openai보다 성능이 훨씬 좋다

In [37]:
import os
upstage_api_key = os.getenv("UPSTAGE_API_KEY")

upstage_client = OpenAI(
    api_key=upstage_api_key,
    base_url="https://api.upstage.ai/v1"
)

In [47]:
up_king_response = upstage_client.embeddings.create(
    input="king",
    model="embedding-query"
)

In [48]:
up_king_vector = np.array(up_king_response.data[0].embedding)
print(up_king_vector.shape)
print(up_king_vector)

(4096,)
[-0.01187134 -0.02058411 -0.00674438 ... -0.01082611  0.00244713
  0.01517487]


In [49]:
up_queen_response = upstage_client.embeddings.create(
    input="queen",
    model="embedding-query"
)
up_queen_vector = np.array(up_queen_response.data[0].embedding)
print(up_queen_vector.shape)

(4096,)


In [50]:
cosine_similarity_up = cosine_similarity(up_king_vector, up_queen_vector)
print(f"Upstage King과 Queen의 유사도: {cosine_similarity_up:.4f}")

Upstage King과 Queen의 유사도: 0.6279


In [51]:
up_kor_king_response = upstage_client.embeddings.create(
    input="왕",
    model="embedding-query"
)
up_kor_king_vector = np.array(up_kor_king_response.data[0].embedding)
print(up_king_vector.shape)

(4096,)


In [53]:
cosine_similarity(up_king_vector, up_kor_king_vector)

np.float64(0.8521901935963604)